## 0. Imports

In [75]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [76]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [77]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

## 1. Prepare data

In [78]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

print("Data fetched successfully!")
print(f"Dataframe shape (rows, columns): {df.shape}\n")

# Display first 5 rows
display(df.head())

Data fetched successfully!
Dataframe shape (rows, columns): (10000, 14)



,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [80]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [81]:
# Data leakage safety
df = df.drop(columns=['UDI', 'Product ID'])

# Deleting this data is essential because model cannot know
# which type of failure is going to happen
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df = df.drop(columns=failure_types)

In [82]:
# Temperature difference
# A machine that dissipates heat poorly is more likely to break down.
df['Temperature_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

# Power
# High load at high speed means a greater risk of failure.
df['Power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']

# Renaming columns for easier operating
df.columns = [col.replace(' ', '_').replace('[', '').replace(']', '') for col in df.columns]

print("New features added: 'Temperature_diff' and 'Power'.")

New features added: 'Temperature_diff' and 'Power'.


In [83]:
# Converting the 'Type' column to numeric values ​​(One-Hot Encoding)
df = pd.get_dummies(df, columns=['Type'], drop_first=True, dtype=int)

# Separating features (X) from label (y)
y = df['Machine_failure']
X = df.drop(columns=['Machine_failure'])

# Class distribution
print("\nClass distribution (0 - works, 1 - failure):")
print(y.value_counts())
print(f"\nPercentage of failures in the set: {(y.sum() / len(y)) * 100:.2f}%")


Class distribution (0 - works, 1 - failure):
Machine_failure
0    9661
1     339
Name: count, dtype: int64

Percentage of failures in the set: 3.39%


In [84]:
display(df)

,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min,Machine_failure,Temperature_diff,Power,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,10.5,66382.8,0,1
1,298.2,308.7,1408,46.3,3,0,10.5,65190.4,1,0
2,298.1,308.5,1498,49.4,5,0,10.4,74001.2,1,0
3,298.2,308.6,1433,39.5,7,0,10.4,56603.5,1,0
4,298.2,308.7,1408,40.0,9,0,10.5,56320.0,1,0
...,...,...,...,...,...,...,...,...,...,...
9995,298.8,308.4,1604,29.5,14,0,9.6,47318.0,0,1
9996,298.9,308.4,1632,31.8,17,0,9.5,51897.6,0,0
9997,299.0,308.6,1645,33.4,22,0,9.6,54943.0,0,1
9998,299.0,308.7,1408,48.5,25,0,9.7,68288.0,0,0


## 2. Baseline

In [85]:
print("Baseline model (Random Forest)")

# Set split with stratify param (fundamental for imbalaced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

# Model training
pipeline.fit(X_train, y_train)

# Prediction on the test set
y_pred = pipeline.predict(X_test)

# Results
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

Baseline model (Random Forest)

=== CONFUSION MATRIX ===
[[1929    3]
 [  18   50]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.94      0.74      0.83        68

    accuracy                           0.99      2000
   macro avg       0.97      0.87      0.91      2000
weighted avg       0.99      0.99      0.99      2000



## 3. Deep Learning (PyTorch)

In [86]:
print("PyTorch version:", torch.__version__)

# Dataset class
class MachineFailureDataset(Dataset):
    def __init__(self, X_data, y_data, scaler=None, is_train=True):
        # Data scaling - we train the scaler only on the training set
        if is_train:
            self.scaler = StandardScaler()
            X_scaled = self.scaler.fit_transform(X_data)
        else:
            # For the test set we only use the previously adjusted scaler
            X_scaled = scaler.transform(X_data)

        # PyTorch tensors
        self.X = torch.tensor(X_scaled, dtype=torch.float32)
        self.y = torch.tensor(y_data.values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Datasets initialization
train_dataset = MachineFailureDataset(X_train, y_train, is_train=True)
test_dataset = MachineFailureDataset(X_test, y_test, scaler=train_dataset.scaler, is_train=False)

# DataLoaders initialization
# We use batch_size=64
# (the network will update the weights after processing 64 examples at a time)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Checking the shape of the data in the first batch
for X_batch, y_batch in train_loader:
    print(f"\nShape of X_batch (Features): {X_batch.shape} -> (batch_size, num_features)")
    print(f"Shape of y_batch (Labels): {y_batch.shape} -> (batch_size, 1)")
    break

PyTorch version: 2.10.0+cpu

Shape of X_batch (Features): torch.Size([64, 9]) -> (batch_size, num_features)
Shape of y_batch (Labels): torch.Size([64, 1]) -> (batch_size, 1)


In [87]:
print("Neural network architecture (Deep Learning)")

# We define the network architecture (inheritance from nn.Module)
class PredictiveMaintenanceNN(nn.Module):
    def __init__(self, input_size):
        super(PredictiveMaintenanceNN, self).__init__()
        # Input layer (9 input features -> 32 output neurons)
        self.layer1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU() # activation function

        # Hidden layer (32 -> 16)
        self.layer2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()

        # Output ayer (16 -> 1 result: 0 or 1)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        # We define the data flow step by step
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output(x)
        # Do not add Sigmoid at the end!
        return x

# Model initialization
input_neurons = 9
model = PredictiveMaintenanceNN(input_neurons)
print(model)

# Weight calculation for the rare class
num_negatives = (y_train == 0).sum()
num_positives = (y_train == 1).sum()
pos_weight = torch.tensor([num_negatives / num_positives], dtype=torch.float32)

# Loss Function and Optimizer
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001) # Learning rate

Neural network architecture (Deep Learning)
PredictiveMaintenanceNN(
  (layer1): Linear(in_features=9, out_features=32, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=32, out_features=16, bias=True)
  (relu2): ReLU()
  (output): Linear(in_features=16, out_features=1, bias=True)
)


In [88]:
epochs = 100

for epoch in range(epochs):
    # Training mode
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        # A. Resetting the gradients from the previous step
        optimizer.zero_grad()

        # B. Forward pass
        predictions = model(X_batch)

        # C. Calculating Loss based on our class weights (pos_weight)
        loss = criterion(predictions, y_batch)

        # D. Backward pass
        loss.backward()

        # E. Optimizer updating weights
        optimizer.step()

        train_loss += loss.item()

    # Printing progress for every tenth epoch
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}")

print("\nTraining completed! Time to evaluate\n")

# Evaluation on the test set
# We switch the model to evaluation mode (stability)
model.eval()

y_true = []
y_pred_list = []

# torch.no_grad() disables gradient tracking (saves memory and speeds up prediction)
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # Logits
        raw_outputs = model(X_batch)

        # Since we used BCEWithLogitsLoss in training, we now need to manually
        # pass the results through Sigmoid to get probabilities (between 0 and 1)
        probabilities = torch.sigmoid(raw_outputs)

        # Converting probabilities to zeros and ones (cutoff threshold = 0.5)
        predictions_binary = (probabilities > 0.5).float()

        # Results
        y_true.extend(y_batch.cpu().numpy())
        y_pred_list.extend(predictions_binary.cpu().numpy())

# Converting lists to flat numpy arrays
y_true = np.array(y_true).flatten()
y_pred_list = np.array(y_pred_list).flatten()

print("\n=== CONFUSION MATRIX (PyTorch) ===")
print(confusion_matrix(y_true, y_pred_list))

print("\n=== CLASSIFICATION REPORT (PyTorch) ===")
print(classification_report(y_true, y_pred_list))

Epoch [10/100], Loss: 0.5308
Epoch [20/100], Loss: 0.4091
Epoch [30/100], Loss: 0.3714
Epoch [40/100], Loss: 0.3424
Epoch [50/100], Loss: 0.3210
Epoch [60/100], Loss: 0.3008
Epoch [70/100], Loss: 0.2865
Epoch [80/100], Loss: 0.2757
Epoch [90/100], Loss: 0.2671
Epoch [100/100], Loss: 0.2578

Training completed! Time to evaluate


=== CONFUSION MATRIX (PyTorch) ===
[[1798  134]
 [   8   60]]

=== CLASSIFICATION REPORT (PyTorch) ===
              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96      1932
         1.0       0.31      0.88      0.46        68

    accuracy                           0.93      2000
   macro avg       0.65      0.91      0.71      2000
weighted avg       0.97      0.93      0.94      2000

